# Kaggle Abalone Submission Notebook

It will:
- load the Kaggle competition files
- build at least two regression models
- compare validation RMSE
- fit the best model on the full training set
- create a `submission.csv` file ready for Kaggle upload

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import root_mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

In [ ]:
# Update this only if your Kaggle competition folder name differs
DATA_DIR = Path("/kaggle/input/playground-series-s4e4")

train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

print("Train exists:", train_path.exists(), train_path)
print("Test exists:", test_path.exists(), test_path)
print("Sample submission exists:", sample_path.exists(), sample_path)

if not train_path.exists():
    raise FileNotFoundError(
        "Could not find train.csv. Check the Kaggle input folder name and update DATA_DIR."
    )

In [ ]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)

display(train_df.head())

In [ ]:
TARGET = "Rings"
ID_COL = "id"

X = train_df.drop(columns=[TARGET])
y = train_df[TARGET].copy()
X_test = test_df.copy()

print("Target:", TARGET)
print("Feature shape:", X.shape)
print("Test feature shape:", X_test.shape)

In [ ]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=400,
        min_samples_leaf=20,
        random_state=42,
    ),
}

results = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_valid)
    rmse = root_mean_squared_error(y_valid, preds)
    results.append({"model": name, "validation_rmse": rmse})
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("validation_rmse").reset_index(drop=True)
display(results_df)

In [ ]:
top_models = results_df["model"].head(2).tolist()
cv_results = []

for name in top_models:
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", models[name]),
    ])
    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    cv_results.append({
        "model": name,
        "cv_rmse_mean": -scores.mean(),
        "cv_rmse_std": scores.std(),
    })

cv_results_df = pd.DataFrame(cv_results).sort_values("cv_rmse_mean").reset_index(drop=True)
display(cv_results_df)

In [ ]:
best_model_name = results_df.loc[0, "model"]
best_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", models[best_model_name]),
])

print("Best model selected:", best_model_name)

In [ ]:
best_model.fit(X, y)
test_preds = best_model.predict(X_test)

print("Predictions generated:", len(test_preds))
print("First 10 predictions:", test_preds[:10])

In [ ]:
submission = sample_submission.copy()
submission["Rings"] = test_preds

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

display(submission.head())
print(f"Saved submission file to: {submission_path}")